# Saved Predictions Review

Summary notebook for predictions we actually saved under `models/predictions/<event_date>/predictions.csv`.

This is different from a retroactive backtest. It only evaluates fights where we already had a saved prediction file, then compares those saved predictions against the actual results currently available in `data/fights.csv`.

Inputs:
- `models/predictions/*/predictions.csv` — saved predictions
- `data/fights.csv` — actual fight outcomes
- `data/events.csv` — event names/dates

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREDICTIONS_DIR = REPO_ROOT / "models" / "predictions"
FIGHTS_PATH = REPO_ROOT / "data" / "fights.csv"
EVENTS_PATH = REPO_ROOT / "data" / "events.csv"

PREDICTIONS_DIR, FIGHTS_PATH, EVENTS_PATH

## 1. Load Saved Predictions and Actual Results

In [ ]:
def read_clean_csv(path, **kwargs):
    df = pd.read_csv(path, **kwargs)
    if len(df) and df.columns[0] in df.columns:
        df = df[df[df.columns[0]].astype(str) != df.columns[0]].copy()
    return df

prediction_files = sorted(PREDICTIONS_DIR.glob("*/predictions.csv"))
if not prediction_files:
    raise FileNotFoundError(f"No saved prediction files found under {PREDICTIONS_DIR}")

prediction_frames = []
for path in prediction_files:
    df = read_clean_csv(path, dtype=str)
    df["prediction_file"] = str(path.relative_to(REPO_ROOT))
    df["prediction_event_date"] = path.parent.name
    prediction_frames.append(df)

preds = pd.concat(prediction_frames, ignore_index=True)
preds["calibrated_prob_f1"] = pd.to_numeric(preds["calibrated_prob_f1"], errors="coerce")
preds["predicted_prob_f1"] = pd.to_numeric(preds["predicted_prob_f1"], errors="coerce")
preds["event_date"] = pd.to_datetime(preds["event_date"], errors="coerce")
preds["scored_at"] = pd.to_datetime(preds["scored_at"], errors="coerce")

# Keep the latest saved prediction per fight if a fight appears in multiple files.
preds = preds.sort_values(["fight_id", "scored_at"]).drop_duplicates("fight_id", keep="last")

fights = read_clean_csv(FIGHTS_PATH, dtype=str)
events = read_clean_csv(EVENTS_PATH, dtype=str)

fights = fights.sort_values("scraped_at", na_position="first").drop_duplicates("fight_id", keep="last")
events = events.sort_values("scraped_at", na_position="first").drop_duplicates("event_id", keep="last")

actual_cols = ["fight_id", "event_id", "fighter_1_outcome", "fighter_2_outcome", "event_status"]
event_cols = ["event_id", "name", "date_formatted"]

review = preds.merge(fights[actual_cols], on="fight_id", how="left")
review = review.merge(events[event_cols], on="event_id", how="left")
review = review.rename(columns={"name": "event_name", "date_formatted": "actual_event_date"})
review["display_event_name"] = review["event_name"].fillna(
    "Unmatched saved prediction (" + review["prediction_event_date"].astype(str) + ")"
)

review["actual_label"] = np.select(
    [
        (review["fighter_1_outcome"] == "W") & (review["fighter_2_outcome"] == "L"),
        (review["fighter_1_outcome"] == "L") & (review["fighter_2_outcome"] == "W"),
    ],
    [1, 0],
    default=np.nan,
)
review["predicted_label"] = np.where(review["calibrated_prob_f1"] >= 0.5, 1, 0)
review["predicted_winner_name"] = np.where(review["predicted_label"] == 1, review["fighter_1_name"], review["fighter_2_name"])
review["actual_winner_name"] = np.select(
    [review["actual_label"] == 1, review["actual_label"] == 0],
    [review["fighter_1_name"], review["fighter_2_name"]],
    default=np.nan,
)
review["resolved"] = review["actual_label"].notna()
review.loc[~review["resolved"], "actual_winner_name"] = "Pending / no W-L result"
review["correct"] = np.where(review["resolved"], review["predicted_label"] == review["actual_label"], np.nan)
review["result_status"] = np.select(
    [
        review["resolved"],
        review["event_id"].isna(),
        review["event_status"].eq("upcoming"),
    ],
    ["resolved", "not in current fights.csv", "pending/upcoming"],
    default="pending/no W-L result",
)
review["predicted_prob_winner"] = np.where(
    review["predicted_label"] == 1,
    review["calibrated_prob_f1"],
    1 - review["calibrated_prob_f1"],
)

print(f"Saved prediction files: {len(prediction_files):,}")
print(f"Unique predicted fights: {len(review):,}")
print(f"Resolved with actual W/L result: {int(review['resolved'].sum()):,}")
print(f"Pending / no result yet: {int((~review['resolved']).sum()):,}")

## 2. Overall Saved-Prediction Accuracy

In [ ]:
resolved = review[review["resolved"]].copy()

if resolved.empty:
    print("No saved predictions have resolved W/L results yet.")
else:
    eps = 1e-15
    p = resolved["calibrated_prob_f1"].clip(eps, 1 - eps)
    y = resolved["actual_label"].astype(int)
    overall = pd.DataFrame([
        {
            "saved_predictions": len(review),
            "resolved_predictions": len(resolved),
            "correct": int(resolved["correct"].sum()),
            "accuracy": resolved["correct"].mean(),
            "log_loss": float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()),
            "brier_score": float(((p - y) ** 2).mean()),
            "avg_predicted_winner_prob": resolved["predicted_prob_winner"].mean(),
        }
    ])
    display(overall.style.format({
        "accuracy": "{:.1%}",
        "log_loss": "{:.4f}",
        "brier_score": "{:.4f}",
        "avg_predicted_winner_prob": "{:.1%}",
    }))

## 3. Event-Level Summary

In [ ]:
event_summary = review.groupby(["event_date", "display_event_name"], dropna=False).agg(
    predicted_fights=("fight_id", "count"),
    resolved_fights=("resolved", "sum"),
    correct=("correct", lambda s: int(pd.Series(s).dropna().sum())),
    accuracy=("correct", lambda s: pd.Series(s).dropna().mean()),
    avg_confidence=("predicted_prob_winner", "mean"),
).reset_index().rename(columns={"display_event_name": "event_name"}).sort_values("event_date", ascending=False)

event_summary_display = event_summary.copy()
event_summary_display["status"] = np.where(event_summary_display["resolved_fights"] > 0, "resolved", "pending / unmatched")
event_summary_display["accuracy"] = np.where(
    event_summary_display["resolved_fights"] > 0,
    event_summary_display["accuracy"].map(lambda v: f"{v:.1%}"),
    "pending",
)
event_summary_display["avg_confidence"] = event_summary_display["avg_confidence"].map(lambda v: f"{v:.1%}")
event_summary_display["event_date"] = event_summary_display["event_date"].dt.strftime("%Y-%m-%d")
event_summary_display[[
    "event_date", "event_name", "status", "predicted_fights", "resolved_fights",
    "correct", "accuracy", "avg_confidence",
]]

## 4. Per-Fight Comparison

In [ ]:
comparison_cols = [
    "event_date", "display_event_name", "fighter_1_name", "fighter_2_name", "weight_class",
    "calibrated_prob_f1", "predicted_winner_name", "actual_winner_name",
    "confidence_tier", "correct", "result_status", "prediction_file",
]

resolved.sort_values(["event_date", "weight_class"])[comparison_cols].style.format({
    "calibrated_prob_f1": "{:.1%}",
})

## 5. Pending Predictions

In [ ]:
pending = review[~review["resolved"]].sort_values(["event_date", "fighter_1_name"])

pending_cols = [
    "event_date", "display_event_name", "fighter_1_name", "fighter_2_name", "weight_class",
    "calibrated_prob_f1", "predicted_winner_name", "confidence_tier", "result_status", "prediction_file",
]

pending[pending_cols].style.format({"calibrated_prob_f1": "{:.1%}"})

## 6. Confidence Tier Performance

In [ ]:
if resolved.empty:
    print("No resolved predictions yet.")
else:
    tier_order = ["high", "medium", "toss-up"]
    tier_summary = resolved.groupby("confidence_tier").agg(
        fights=("fight_id", "count"),
        accuracy=("correct", "mean"),
        avg_confidence=("predicted_prob_winner", "mean"),
    ).reindex(tier_order).reset_index()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(tier_summary["confidence_tier"], tier_summary["accuracy"])
    ax.axhline(resolved["correct"].mean(), color="black", linestyle="--", linewidth=1, label="overall")
    ax.set_title("Saved Predictions: Accuracy by Confidence Tier")
    ax.set_ylabel("Accuracy")
    ax.legend()
    plt.show()

    display(tier_summary.style.format({"accuracy": "{:.1%}", "avg_confidence": "{:.1%}"}))

## 7. Accuracy Over Saved Events

In [ ]:
plot_df = event_summary[event_summary["resolved_fights"] > 0].sort_values("event_date").copy()

if plot_df.empty:
    print("No resolved events to plot yet.")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(plot_df["event_date"], plot_df["accuracy"], marker="o", linewidth=2)
    ax.axhline(resolved["correct"].mean(), color="black", linestyle="--", linewidth=1, label="overall")
    ax.set_title("Saved Prediction Accuracy by Event")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 8. Search One Event

In [ ]:
# Change this to inspect one saved card, e.g. "2026-05-09" or part of an event name.
EVENT_FILTER = ""

if EVENT_FILTER.strip():
    text = EVENT_FILTER.strip().lower()
    mask = (
        review["event_date"].astype(str).str.lower().str.contains(text, na=False)
        | review["display_event_name"].astype(str).str.lower().str.contains(text, na=False)
    )
    event_rows = review[mask].sort_values(["event_date", "fighter_1_name"])
    display(event_rows[comparison_cols].style.format({"calibrated_prob_f1": "{:.1%}"}))
else:
    print("Set EVENT_FILTER to inspect a specific saved event.")